In [1]:
from energycast.utils import data_utils

from energycast.features import calendar_features
from energycast.features import statistical_features
from energycast.features import weather_features

from energycast.normalization_and_preprocessing import norm_utils

In [ ]:
from importlib import reload

reload(data_utils)

<module 'energycast.utils.data_utils' from '/home/undb/projects/EnergyCast/src/energycast/utils/data_utils.py'>

In [ ]:
df_sm = data_utils.load_parquet("raw", "smart_meter_readings")
df_id = data_utils.load_parquet("raw", "smart_meter_metadata")
df_weather = data_utils.load_parquet("raw", "weather")

### Column selection for minimal consumption prediction dataset

In [ ]:
test_customers = ("43Z-STO00711198D", "43Z-STO006304393")
df_sm_test = df_sm.loc[df_sm["object_id"].isin(test_customers)]


df_sm_test = df_sm_test[["timestamp", "object_id", "energy_import_kwh"]]
df_sm_test = df_sm_test.rename(columns={"energy_import_kwh": "energy_kwh"})
df_sm_test = data_utils.downcast_float_in_df(df_sm_test, columns_to_exclude=[])

df_sm_test

/home/undb/projects/EnergyCast/.venv/lib/python3.12/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,timestamp,object_id,energy_kwh
0,2017-04-01 01:00:00,43Z-STO00711198D,16.328125
1,2017-04-01 02:00:00,43Z-STO00711198D,15.507812
2,2017-04-01 03:00:00,43Z-STO00711198D,12.718750
3,2017-04-01 04:00:00,43Z-STO00711198D,8.531250
4,2017-04-01 05:00:00,43Z-STO00711198D,8.859375
...,...,...,...
5234491,2020-03-31 20:00:00,43Z-STO006304393,1.578125
5234492,2020-03-31 21:00:00,43Z-STO006304393,1.845703
5234493,2020-03-31 22:00:00,43Z-STO006304393,0.632324
5234494,2020-03-31 23:00:00,43Z-STO006304393,0.385742


In [ ]:
df_sm_test = statistical_features.add_rolling_stats(
    df_sm_test, windows=[6, 12, 24], aggs=["mean", "std"]
)
df_sm_test = statistical_features.add_lagged_energy_values(
    df_sm_test, lags=[1, 6, 12, 24, 168]
)
df_sm_test = data_utils.downcast_float_in_df(df_sm_test, columns_to_exclude=[])
df_sm_test = statistical_features.freeze_history_features(df_sm_test, 24, [])
df_sm_test = statistical_features.add_customer_averages(df_sm_test)
df_sm_test = data_utils.downcast_float_in_df(df_sm_test, columns_to_exclude=[])

df_sm_test = calendar_features.add_calendar_features(df_sm_test)
df_sm_test = data_utils.downcast_float_in_df(df_sm_test, columns_to_exclude=[])

df_sm_test = weather_features.join_weather(
    df=df_sm_test, df_id=df_id, df_weather=df_weather
)
df_sm_test = data_utils.downcast_float_in_df(df_sm_test, columns_to_exclude=[])

In [ ]:
df_sm_test = norm_utils.normalize_bool_columns(df_sm_test)
df_sm_test = norm_utils.normalize_categorical_columns(
    df_sm_test, categorical_cols=["weekday", "hour"]
)

In [9]:
splits, name = data_utils.load_splits("default_24")

In [15]:
splits_dfs = data_utils.prepare_splits(df=df_sm_test, splits=splits)

In [ ]:
data_utils.save_splits(splits_dfs, name)

[PosixPath('/home/undb/projects/EnergyCast/data/processed/default_24_train.parquet'),
 PosixPath('/home/undb/projects/EnergyCast/data/processed/default_24_dev.parquet'),
 PosixPath('/home/undb/projects/EnergyCast/data/processed/default_24_test.parquet')]